# Notebook 05-i: Fermion Doubling and the Wilson Term

**Learning objectives:**
- Understand why naive lattice fermions produce 16 species instead of 1
- Visualize the free-field dispersion relation in momentum space
- See how the Wilson term lifts doublers
- Understand the cost: explicit chiral symmetry breaking

**Prerequisites:** Notebook 05 (Wilson-Dirac operator)

**Supplements:** This notebook develops the fermion doubling discussion from Notebook 05.

In [ ]:
from notebook_utils import setup_paths
setup_paths()

import numpy as np
import matplotlib.pyplot as plt
from MesonBase import build_wilson_dirac_matrix, generate_identity_gauge_field

## 1. The Doubling Problem

In the continuum, the Dirac equation for a free fermion has a single
dispersion relation: $E^2 = \vec{p}^2 + m^2$.

On the lattice, replacing $\partial_\mu \to \frac{1}{2a}(\delta_{+\mu} - \delta_{-\mu})$
leads to $\sin(p_\mu a)/a$ instead of $p_\mu$. The sine function has
zeros not only at $p_\mu = 0$ (the physical mode) but also at
$p_\mu = \pi/a$ (the boundary of the Brillouin zone).

In 4D, each of the 4 momentum components can independently be $0$ or $\pi/a$,
giving $2^4 = 16$ massless modes — the **doublers**.

## 2. Free-Field Dispersion Relation

For the naive lattice Dirac operator (Wilson $r=0$), the energy-momentum
relation in 1D is:

$$E_{\rm naive}(p) = \arcsin\bigl[\sin(pa)\bigr]/a$$

With the Wilson term ($r > 0$), the dispersion becomes:

$$E_W(p) \approx \sqrt{\sin^2(pa)/a^2 + \bigl[\frac{r}{a}(1 - \cos pa)\bigr]^2 + m^2}$$

The Wilson term adds a mass $\sim 2r/a$ at $p = \pi/a$, lifting the doubler.

In [ ]:
# 1D dispersion relation: naive vs Wilson
a = 1.0  # lattice spacing
p = np.linspace(-np.pi/a, np.pi/a, 500)
m = 0.1

# Naive (r=0): E^2 = sin^2(p) + m^2
E_naive = np.sqrt(np.sin(p*a)**2 / a**2 + m**2)

# Wilson r=0.5
r = 0.5
wilson_mass = (r/a) * (1 - np.cos(p*a))
E_wilson_05 = np.sqrt(np.sin(p*a)**2 / a**2 + (m + wilson_mass)**2)

# Wilson r=1.0
r = 1.0
wilson_mass = (r/a) * (1 - np.cos(p*a))
E_wilson_10 = np.sqrt(np.sin(p*a)**2 / a**2 + (m + wilson_mass)**2)

# Continuum
E_cont = np.sqrt(p**2 + m**2)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(p, E_cont, 'k-', linewidth=2, label='Continuum')
ax.plot(p, E_naive, 'b--', linewidth=1.5, label='Naive (r=0)')
ax.plot(p, E_wilson_05, 'r-', linewidth=1.5, label='Wilson r=0.5')
ax.plot(p, E_wilson_10, 'g-.', linewidth=1.5, label='Wilson r=1.0')

ax.set_xlabel('Momentum p', fontsize=13)
ax.set_ylabel('Energy E(p)', fontsize=13)
ax.set_title('Free-field dispersion relation (1D)')
ax.axvline(0, color='gray', ls=':', alpha=0.3)
ax.axvline(np.pi, color='gray', ls=':', alpha=0.3, label=r'$p = \pi/a$')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("At p=0:  all agree (physical mode)")
print(f"At p=pi: naive E={E_naive[0]:.2f} (doubler!), Wilson r=1 E={E_wilson_10[0]:.2f} (lifted)")

## 3. Counting Doublers on the Lattice

Let's directly count the near-zero eigenvalues of the Dirac operator
for different Wilson parameter values. On a small lattice, we can
diagonalize $D_W$ and count modes with small real part.

In [ ]:
La_small = [2, 2, 2, 2]
U_small, _ = generate_identity_gauge_field(La_small)
mass = 0.1

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, r_val in zip(axes, [0.0, 0.5, 1.0]):
    D = build_wilson_dirac_matrix(mass, La_small, wilson_r=r_val, U=U_small)
    eigs = np.linalg.eigvals(D.toarray())
    
    ax.scatter(eigs.real, eigs.imag, s=8, alpha=0.5)
    ax.axvline(mass + 4*r_val, color='r', ls='--', alpha=0.5,
               label=f'$m + 4r = {mass + 4*r_val:.1f}$')
    ax.set_xlabel('Re($\\lambda$)')
    ax.set_ylabel('Im($\\lambda$)')
    ax.set_title(f'r = {r_val}')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    # Count near-zero modes (|Re(lambda)| < threshold)
    threshold = 0.5
    n_near_zero = np.sum(np.abs(eigs.real) < threshold)
    print(f"r = {r_val}: {n_near_zero} eigenvalues with |Re(lambda)| < {threshold}")

plt.suptitle('Eigenvalue spectra: doubler removal by Wilson term', fontsize=13)
plt.tight_layout()
plt.show()

## 4. The Brillouin Zone and Doubler Locations

In 4D, the doublers sit at the 15 non-trivial corners of the Brillouin zone
$\mathcal{B} = [-\pi/a, \pi/a]^4$. The Wilson term gives them effective
masses proportional to the number of momentum components at $\pi/a$:

| Corner type | Count | Effective mass at $r=1$ |
|-------------|-------|------------------------|
| $(0,0,0,0)$ | 1 | $m$ (physical) |
| $(\pi,0,0,0)$ etc. | $\binom{4}{1} = 4$ | $m + 2r/a$ |
| $(\pi,\pi,0,0)$ etc. | $\binom{4}{2} = 6$ | $m + 4r/a$ |
| $(\pi,\pi,\pi,0)$ etc. | $\binom{4}{3} = 4$ | $m + 6r/a$ |
| $(\pi,\pi,\pi,\pi)$ | 1 | $m + 8r/a$ |

Total: $1 + 4 + 6 + 4 + 1 = 16$ species, but only 1 remains light.

In [ ]:
# Verify: count doublers by type
from itertools import product

doubler_types = {}
for corner in product([0, 1], repeat=4):
    n_pi = sum(corner)
    if n_pi not in doubler_types:
        doubler_types[n_pi] = 0
    doubler_types[n_pi] += 1

print("Doubler classification in 4D:")
print(f"{'# components at pi/a':>22s} | {'Count':>5s} | {'Wilson mass shift':>16s}")
print("-" * 50)
for n_pi in sorted(doubler_types):
    count = doubler_types[n_pi]
    shift = 2 * n_pi  # for r=1
    label = "physical" if n_pi == 0 else f"2r x {n_pi} = {shift}"
    print(f"{n_pi:>22d} | {count:>5d} | {label:>16s}")
print(f"{'Total':>22s} | {sum(doubler_types.values()):>5d} |")

## 5. The Cost: Chiral Symmetry Breaking

The Wilson term explicitly breaks chiral symmetry. Consequences:

1. **Additive mass renormalization**: Even at $m_{\rm bare} = 0$, the quark
   acquires an effective mass $m_{\rm crit}(\beta) \neq 0$.
   The physical quark mass is $m_q = m_{\rm bare} - m_{\rm crit}$.

2. **No exact chiral limit**: $m_q = 0$ requires fine-tuning $m_{\rm bare}$
   to the critical value, which depends on $\beta$ and must be determined
   non-perturbatively (see Notebook 07-i).

3. **$O(a)$ discretization errors**: Wilson fermions have $O(a)$ artifacts
   (vs $O(a^2)$ for chirally symmetric formulations).

## 6. Alternative Fermion Formulations

The doubling problem has motivated several alternative approaches:

| Formulation | Doublers | Chiral symmetry | Cost |
|------------|---------|-----------------|------|
| **Wilson** | None (lifted) | Broken | Additive renormalization, $O(a)$ errors |
| **Staggered** | 4 (reduced from 16) | Partial ($U(1)$) | "Rooting" controversy |
| **Domain-wall** | None | Approximate (exponentially good) | 5th dimension adds cost |
| **Overlap** | None | Exact (Ginsparg-Wilson) | Very expensive ($\sim 10\times$) |

Our code uses Wilson fermions — the simplest and most transparent approach.

## Exercises

1. **Dispersion relation**: Modify the 1D dispersion plot to show 2D.
   Plot $E(p_x, p_y)$ as a surface or contour plot for naive and Wilson
   fermions. Where are the doubler minima?

2. **Counting eigenvalues**: On a $2^4$ lattice, vary $r$ from 0 to 1
   in steps of 0.1. At each $r$, count eigenvalues with
   $|\mathrm{Re}(\lambda)| < 0.3$. Plot the count vs $r$ — at what $r$
   do the doublers separate from the physical mode?